<a href="https://colab.research.google.com/github/NileshPatil24-a/Deep_Learning/blob/main/age_gender_revied.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
 76% 252M/331M [00:00<00:00, 694MB/s] 
100% 331M/331M [00:00<00:00, 715MB/s]


In [5]:
import zipfile

zip_ref = zipfile.ZipFile('/content/utkface-new.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()


In [6]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [8]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [9]:
len(age)

23708

In [10]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [11]:
df.shape

(23708, 3)

In [12]:
df.head()

,age,gender,img
0,26,0,26_0_1_20170116153040656.jpg.chip.jpg
1,23,1,23_1_2_20170116172848833.jpg.chip.jpg
2,50,0,50_0_0_20170117190657761.jpg.chip.jpg
3,9,0,9_0_0_20170110220413289.jpg.chip.jpg
4,40,1,40_1_1_20170116223441453.jpg.chip.jpg


In [13]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [14]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [15]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [17]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [18]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [19]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [20]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [21]:
model.fit(train_generator, batch_size=32, epochs=10, validation_data=test_generator)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.

In [22]:
# Define the output signature for the generator
output_signature = (
    tf.TensorSpec(shape=(None, 200, 200, 3), dtype=tf.float32),  # Images
    (
        tf.TensorSpec(shape=(None, 1), dtype=tf.float32),  # Age output
        tf.TensorSpec(shape=(None, 1), dtype=tf.float32)   # Gender output
    )
)

# Create tf.data.Dataset from the generators
train_dataset = tf.data.Dataset.from_generator(
    lambda: (
        (
            tf.constant(x, dtype=tf.float32),
            (
                tf.constant(np.expand_dims(y_age, -1), dtype=tf.float32),
                tf.constant(np.expand_dims(y_gender, -1), dtype=tf.float32)
            )
        )
        for x, [y_age, y_gender] in train_generator
    ),
    output_signature=output_signature
)

test_dataset = tf.data.Dataset.from_generator(
    lambda: (
        (
            tf.constant(x, dtype=tf.float32),
            (
                tf.constant(np.expand_dims(y_age, -1), dtype=tf.float32),
                tf.constant(np.expand_dims(y_gender, -1), dtype=tf.float32)
            )
        )
        for x, [y_age, y_gender] in test_generator
    ),
    output_signature=output_signature
)

# Now call model.fit with the created datasets
model.fit(train_dataset, epochs=10, validation_data=test_dataset)

Epoch 1/10
    348/Unknown 138s 352ms/step - age_loss: 16.4888 - age_mae: 16.4888 - gender_accuracy: 0.5025 - gender_loss: 1.8254 - loss: 197.2046

KeyboardInterrupt: 